# Kayak : où partir en France ? Collecte, Data Lake, ETL et Data Warehouse

**Contexte.** L'équipe marketing de Kayak veut recommander des destinations et des hôtels à partir de **données réelles** (météo, hôtels) pour les **35 meilleures villes à visiter en France**.

**Collecte des données brutes :**
- Nominatim : collecte des coordonnées GPS
- Open-Meteo : météo à 7 jours
- Booking.com : hôtels (scraping)

**Sauvegarde dans le DATA LAKE:**
- Bucket S3 AWS
- Copie locale

**ETL:**
- Pandas 

**Sauvegarde dans le DATA WAREHOUSE:**
- NeonDB
- Copie locale

**Livrables:**
- Un csv enrichi (météo et hôtels) dans le DATA LAKE
- Une base SQL contenant les mêmes données néttoyées
- Deux cartes : top 5 des destinations et top 20 des hôtels

## 1. Prérequis, paramètres et imports

**Ce notebook exécute les trois collectes** (API Nominatim, API Open-Meteo, scraping Booking). Un « Run All » refait donc tout : compter **environ 15 minutes**, dont la majeure partie pour le scraping.

| Prérequis | Détail |
|---|---|
| Internet | Nominatim, Open-Meteo, Booking.com, AWS S3, Neon |
| Google Chrome | piloté par Selenium (la fenêtre s'ouvre et ne doit pas être fermée pendant la collecte) |
| Compte AWS | clés d'accès avec droits sur S3 |
| Projet Neon | base PostgreSQL, créée dans une **région européenne** ; copier sa chaîne de connexion |
| Fichier `.env` | à créer à partir de `.env.example` : `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_BUCKET_NAME`, `AWS_REGION`, `NEON_DATABASE_URL` |


In [ ]:
import io
import os
import random
import re
import time
from datetime import date, timedelta
from pathlib import Path

import boto3
import pandas as pd
import plotly.express as px
import requests
from bs4 import BeautifulSoup
from botocore.exceptions import ClientError
from dotenv import load_dotenv            # lit le fichier .env (secrets)
from sqlalchemy import create_engine, text

load_dotenv(Path(".env"), override=True)                              # charge AWS_* et NEON_DATABASE_URL depuis .env
pd.options.display.max_columns = 30
pd.options.display.width = 140

USER_AGENT = "kayak-project/1.0"           # identification auprès des API

# Liste de 35 villes
CITIES = [
    "Mont Saint Michel", "St Malo", "Bayeux", "Le Havre", "Rouen", "Paris",
    "Amiens", "Lille", "Strasbourg", "Chateau du Haut Koenigsbourg", "Colmar",
    "Eguisheim", "Besancon", "Dijon", "Annecy", "Grenoble", "Lyon",
    "Gorges du Verdon", "Bormes les Mimosas", "Cassis", "Marseille",
    "Aix en Provence", "Avignon", "Uzes", "Nimes", "Aigues Mortes",
    "Saintes Maries de la mer", "Collioure", "Carcassonne", "Ariege",
    "Toulouse", "Montauban", "Biarritz", "Bayonne", "La Rochelle",
]
print(len(CITIES), "villes dans le périmètre")

35 villes dans le périmètre


## 2. Data Lake

Le **Data Lake** conserve les données **brutes**, sans modification. Celui-ci sera structuré en **couches** (`raw/` pour le brut, `enriched/` pour le fichier final) puis par **jeu de données et date de collecte** :

```
raw/cities/date=2026-10-18/cities_coordinates.csv
raw/weather/date=2026-10-18/weather_data.csv
raw/hotels/date=2026-10-18/hotels_data.csv
enriched/kayak_enriched/date=2026-10-18/kayak_enriched.csv
```

- **Sauvegarde simultanée** : chaque fichier est écrit **à la fois** dans le dossier local `data_lake/` et dans un bucket **AWS S3**, avec exactement le même rangement.
- Le bucket est **chiffré** (AES-256) et **versionné** (une relance le même jour ne perd pas l'ancienne version), dans la région de Paris (`eu-west-3`) pour garder les données en Europe.
- **Lecture** : c'est S3 qui fait foi ; la copie locale sert de sauvegarde.


In [ ]:
LAKE_DIR = Path("../data_lake")
LAKE_DIR.mkdir(parents=True, exist_ok=True)         # copie locale du lac
AWS_REGION = os.getenv("AWS_REGION", "eu-west-3")
BUCKET = os.getenv("AWS_BUCKET_NAME")

s3 = boto3.client("s3", region_name=AWS_REGION)     # clés lues dans l'environnement (.env)
try:
    s3.head_bucket(Bucket=BUCKET)                   # Vérification de l'existance du bucket
except ClientError as e:
    status = e.response.get("ResponseMetadata", {}).get("HTTPStatusCode")

    if status == 404:
        print(f"Le bucket '{BUCKET}' n'existe pas.")
    else:
        raise


def lake_write(df, layer, dataset, filename):
    # Dépose un DataFrame (CSV) dans le lac : en local ET dans S3, au même chemin.
    key = f"{layer}/{dataset}/date={date.today().isoformat()}/{filename}"
    path = LAKE_DIR / key
    path.parent.mkdir(parents=True, exist_ok=True)
    
    df.to_csv(path, index=False)                                              # copie locale
    s3.put_object(Bucket=BUCKET, Key=key, ServerSideEncryption="AES256",
                  Body=df.to_csv(index=False).encode("utf-8"))                # copie S3
    
    print(f"  déposé en local et sur S3 : {key}")
    return key


def lake_read(layer, dataset):
    # Relit depuis S3 le fichier le plus récent d'un jeu de données (les dates se trient alphabétiquement).
    prefix = f"{layer}/{dataset}/"
    pages = s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=prefix)
    keys = [o["Key"] for p in pages for o in p.get("Contents", [])]
    return pd.read_csv(io.BytesIO(s3.get_object(Bucket=BUCKET, Key=max(keys))["Body"].read()))

## 3. Données personnelles (RGPD) et conditions d'usage des sources

**RGPD**
- Le projet ne collecte **aucune donnée d'utilisateur** : ni compte, ni adresse IP de visiteur, ni historique de recherche.
- Il ne conserve que des informations **commerciales ou géographiques** : nom, URL, note globale et prix d'un hôtel, coordonnées d'une ville, code météo.
- **Les avis clients sont exclus** (texte et auteurs).
- Point de vigilance : le nom d'une chambre d'hôtes peut désigner une personne physique. C'est une information publiée par l'établissement lui-même, à confirmer avec le DPO avant toute mise en production.

**Conditions d'usage des sources**

| Source | Règle à respecter | Ce que fait le notebook |
|---|---|---|
| Nominatim (OpenStreetMap) | 1 requête/seconde maximum, `User-Agent` identifiant, mention « © contributeurs OpenStreetMap » (licence ODbL) : https://operations.osmfoundation.org/policies/nominatim/ | pause d'1 s entre deux requêtes, `User-Agent` fourni |
| Open-Meteo | API gratuite réservée à un usage **non commercial**, attribution requise | 35 requêtes seulement |
| Booking.com | Les CGU **interdisent le scraping** (voir paragraphe A15.2) https://www.booking.com/content/terms.fr.html | Projet à usage pédagogique uniquement : 20 hôtels par ville, pauses aléatoires, arrêt après 3 échecs, aucun contournement de CAPTCHA, aucune republication. En production : partenariat ou API officielle |

## 4. Collecte des données

Chaque jeu de données est **déposé dans le Data Lake dès la fin de sa collecte** (en local et sur S3) : si une étape suivante échoue, ce qui est déjà collecté n'est pas perdu.

### 4.1 Coordonnées GPS des villes (API Nominatim)

L'API gratuite **Nominatim** (OpenStreetMap, sans clé) transforme un nom de ville en coordonnées. Le `place_id` renvoyé sert d'**identifiant unique de la ville** (`city_id`) pour relier ensuite la météo et les hôtels.

In [5]:
def geocode(city):
    # Interroge Nominatim pour une ville et renvoie un dictionnaire (statut inclus).
    params = {"q": city, "format": "json", "countrycodes": "fr", "limit": 1}
    
    try:
        r = requests.get("https://nominatim.openstreetmap.org/search", params=params,
                         headers={"User-Agent": USER_AGENT}, timeout=10)
        r.raise_for_status()
        data = r.json()
    except (requests.RequestException, ValueError):
        return {"query": city, "status": "request_error"}
    
    if not data:
        return {"query": city, "status": "Non trouvée"}
    
    p = data[0]
    
    return {"query": city, "place_id": p["place_id"], "place_name": p.get("name"),
            "place_lat": float(p["lat"]), "place_lon": float(p["lon"]),
            "display_name": p["display_name"], "status": "Ok"}

In [6]:
rows = []

for city in CITIES:
    rows.append(geocode(city))
    time.sleep(1)                          # politique Nominatim : 1 requête par seconde

cities_raw = pd.DataFrame(rows)

print(cities_raw["status"].value_counts().to_dict())

cities_ok = cities_raw[cities_raw["status"] == "Ok"].reset_index(drop=True)   # seules les villes trouvées continuent

if cities_ok.empty:
    raise RuntimeError("Aucune ville géocodée : vérifier la connexion à Nominatim.")

if len(cities_ok) < len(CITIES):
    print("Attention, villes non géocodées :", cities_raw.loc[cities_raw["status"] != "Ok", "query"].tolist())


lake_write(cities_raw, "raw", "cities", "cities_coordinates.csv")

cities_raw[["query", "place_id", "place_name", "place_lat", "place_lon"]].head()

{'Ok': 35}
  déposé en local et sur S3 : raw/cities/date=2026-11-02/cities_coordinates.csv


,query,place_id,place_name,place_lat,place_lon
0,Mont Saint Michel,283872868,Mont Saint-Michel,48.635954,-1.511460
1,St Malo,284162046,Saint-Malo,48.649518,-2.026041
2,Bayeux,283111531,Bayeux,49.276462,-0.702474
3,Le Havre,99608512,Le Havre,49.493898,0.107973
4,Rouen,430117916,Rouen,49.440459,1.093966


### 4.2 Météo à 7 jours (API Open-Meteo)

Pour chaque ville, **Open-Meteo** (gratuit, sans clé) renvoie un **code météo** par jour pour les 7 prochains jours. Chaque code est converti en **score de 0 à 100** avec une grille simple (100 = ciel dégagé, 0 = orage), puis le **score d'une ville est la moyenne de ses 7 jours**.

> L'énoncé préconnise OpenWeatherMap mais nous utiliserons ici Open-Meteo qui ne nécessite pas la création d'un compte. Limite connue : le score ne tient compte que du code météo (ni température, ni vent).

In [7]:
# Grille code météo (norme WMO) -> score de 0 à 100
WEATHER_SCORE = {
    0: 100, 1: 95, 2: 85, 3: 70,                       # ciel dégagé à couvert
    45: 50, 48: 40,                                    # brouillard
    51: 65, 53: 55, 55: 45, 56: 35, 57: 20,            # bruine
    61: 55, 63: 40, 65: 20, 66: 15, 67: 5,             # pluie
    71: 40, 73: 25, 75: 10, 77: 20,                    # neige
    80: 45, 81: 30, 82: 10, 85: 20, 86: 5,             # averses
    95: 0, 96: 0, 99: 0,                               # orages
}


def forecast_scores(lat, lon):
    # Renvoie 7 scores météo (un par jour) pour des coordonnées, ou 7 None en cas d'erreur.
    params = {"latitude": lat, "longitude": lon, "daily": "weathercode",
              "forecast_days": 7, "timezone": "auto"}
    
    try:
        r = requests.get("https://api.open-meteo.com/v1/forecast", params=params,
                         headers={"User-Agent": USER_AGENT}, timeout=10)
        r.raise_for_status()
        codes = r.json()["daily"]["weathercode"]
    except (requests.RequestException, KeyError, ValueError):
        return [None] * 7
    
    return [WEATHER_SCORE.get(c) for c in codes]

In [8]:
day_cols = [f"weather_day_{i}" for i in range(1, 8)]

scores = [forecast_scores(r.place_lat, r.place_lon) for r in cities_ok.itertuples()]

weather_raw = pd.concat([cities_ok, pd.DataFrame(scores, columns=day_cols)], axis=1)
weather_raw["weather_score"] = weather_raw[day_cols].mean(axis=1)   # moyenne sur 7 jours

if weather_raw["weather_score"].isna().all():
    raise RuntimeError("Aucune prévision reçue")

lake_write(weather_raw, "raw", "weather", "weather_data.csv")

weather_raw[["place_name", "weather_day_1", "weather_day_2", "weather_day_3", "weather_day_4", 
             "weather_day_5", "weather_day_6", "weather_day_7", "weather_score"]] \
    .sort_values("weather_score", ascending=False).head(5)

  déposé en local et sur S3 : raw/weather/date=2026-11-02/weather_data.csv


,place_name,weather_day_1,weather_day_2,weather_day_3,weather_day_4,weather_day_5,weather_day_6,weather_day_7,weather_score
33,Bayonne,70,70,70,85,85,65,70,73.571429
32,Biarritz,70,70,70,85,85,65,70,73.571429
7,Lille,55,55,100,100,70,50,65,70.714286
6,Amiens,55,65,85,85,65,65,65,69.285714
28,Carcassonne,70,70,55,65,70,70,70,67.142857


### 4.3 Hôtels (scraping de Booking.com avec Selenium et BeautifulSoup)

La collecte des données relatives aux hôtels depuis Booking.com nécessite ici un recours au scraping de la page de résultats de chaque ville.

- **Selenium** pilote Chrome, car la page est construite en JavaScript. Le navigateur est **visible** : ne pas fermer la fenêtre pendant la collecte.
- **BeautifulSoup** lit ensuite le HTML et extrait, pour chaque hôtel : nom, URL, note, prix et description.
- On garde **20 hôtels par ville**, avec une pause aléatoire de 3 à 7 s entre deux pages, et on **s'arrête après 3 échecs consécutifs** (blocage probable). À chaque échec, la page reçue est affichée et une capture `debug_booking.png` est enregistrée pour comprendre le blocage (CAPTCHA, page vide, nouvelle mise en page).
- Les coordonnées de l'hôtel ne figurent pas dans la page de résultats : on y met provisoirement celles de la ville (voir §7 pour la correction sur le Top 20).

Les fonctions de lecture du HTML (`parse_score`, `parse_hotels`) sont séparées de la navigation et testables sans navigateur.

In [9]:
def parse_score(texte):
    # Extrait la note d'un texte de carte. Booking répète la note ('7.67.6') :
    # on ne lit donc qu'un chiffre après le séparateur.
    m = re.search(r"(\d[.,]\d|\b10\b)", texte or "")
    return float(m.group(1).replace(",", ".")) if m else None


def parse_hotels(html, max_hotels=20):
    # Extrait les hôtels d'une page de résultats Booking (BeautifulSoup).
    soup = BeautifulSoup(html, "html.parser")
    
    hotels = []
    
    for card in soup.find_all("div", attrs={"data-testid": "property-card"})[:max_hotels]:
        def texte(test_id, tag="div"):
            el = card.find(tag, attrs={"data-testid": test_id})
            return el.get_text(" ", strip=True) if el else None

        lien = card.find("a", attrs={"data-testid": "title-link"})
        hotels.append({
            "hotel_name": texte("title"),
            "hotel_url": lien["href"] if lien and lien.get("href") else None,
            "hotel_score": parse_score(texte("review-score")),
            "hotel_description": texte("recommended-units"),
            "hotel_price": texte("price-and-discounted-price", tag="span"),
        })
    
    return hotels

In [10]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait


def scrape_booking(cities_df, days_ahead=1, nights=1):
    # Scrape les hôtels de chaque ville (séjour simulé : 2 adultes, 1 chambre).
    checkin = date.today() + timedelta(days=days_ahead)
    checkout = checkin + timedelta(days=nights)
    options = webdriver.ChromeOptions()
    options.add_argument("--lang=fr-FR")
    driver = webdriver.Chrome(options=options)   # Chrome visible ; Selenium télécharge le pilote tout seul

    hotels, failures = [], 0
    try:
        for city in cities_df.itertuples():
            url = ("https://www.booking.com/searchresults.fr.html?"
                   f"ss={city.place_name}&checkin={checkin}&checkout={checkout}"
                   "&group_adults=2&no_rooms=1&group_children=0")
            try:
                driver.get(url)
                try:                             # ferme le bandeau de cookies s'il apparaît
                    WebDriverWait(driver, 3).until(EC.element_to_be_clickable(
                        (By.ID, "onetrust-accept-btn-handler"))).click()
                except Exception:
                    pass
                WebDriverWait(driver, 15).until(EC.presence_of_element_located(
                    (By.CSS_SELECTOR, "[data-testid='property-card']")))
                time.sleep(2)                    # laisse le JavaScript finir
                found = parse_hotels(driver.page_source)
            except Exception as err:             # page bloquée, délai dépassé...
                failures += 1
                print(f"Échec {failures}/3 pour {city.place_name} : {type(err).__name__}")
                print("   page reçue :", driver.title, "|", driver.current_url)
                driver.save_screenshot("debug_booking.png")   # à ouvrir : CAPTCHA, page vide, autre mise en page ?
                if failures >= 3:
                    print("Blocage probable : arrêt de la collecte.")
                    break
                continue
            failures = 0
            for h in found:                      # on rattache chaque hôtel à sa ville
                h.update(city_id=city.place_id, city_name=city.place_name,
                         hotel_lat=city.place_lat, hotel_lon=city.place_lon)
            hotels += found
            print(f"{city.place_name} : {len(found)} hôtels")
            time.sleep(random.uniform(3, 7))     # pause de politesse
    finally:
        driver.quit()
    return pd.DataFrame(hotels)

In [11]:
hotels_raw = scrape_booking(cities_ok)           # environ 10 minutes pour 35 villes

if hotels_raw.empty:                             # on ne dépose jamais un fichier vide dans le lac
    raise RuntimeError("Aucun hôtel collecté : voir les messages ci-dessus et debug_booking.png.")
print(f"\n{len(hotels_raw)} hôtels dans {hotels_raw['city_id'].nunique()} villes")
lake_write(hotels_raw, "raw", "hotels", "hotels_data.csv")
hotels_raw[["city_name", "hotel_name", "hotel_score", "hotel_price"]].head(5)

Mont Saint-Michel : 20 hôtels
Saint-Malo : 20 hôtels
Bayeux : 20 hôtels
Le Havre : 20 hôtels
Rouen : 20 hôtels
Paris : 20 hôtels
Amiens : 20 hôtels
Lille : 20 hôtels
Strasbourg : 20 hôtels
Château du Haut-Kœnigsbourg : 20 hôtels
Colmar : 20 hôtels
Eguisheim : 20 hôtels
Besançon : 20 hôtels
Dijon : 20 hôtels
Annecy : 20 hôtels
Grenoble : 20 hôtels
Lyon : 20 hôtels
Gorges du Verdon : 20 hôtels
Bormes-les-Mimosas : 20 hôtels
Cassis : 20 hôtels
Marseille : 20 hôtels
Aix-en-Provence : 20 hôtels
Avignon : 20 hôtels
Uzès : 20 hôtels
Nîmes : 20 hôtels
Aigues-Mortes : 20 hôtels
Saintes-Maries-de-la-Mer : 20 hôtels
Collioure : 20 hôtels
Carcassonne : 20 hôtels
Ariège : 20 hôtels
Toulouse : 20 hôtels
Montauban : 20 hôtels
Biarritz : 20 hôtels
Bayonne : 20 hôtels
La Rochelle : 20 hôtels

700 hôtels dans 35 villes
  déposé en local et sur S3 : raw/hotels/date=2026-11-02/hotels_data.csv


,city_name,hotel_name,hotel_score,hotel_price
0,Mont Saint-Michel,Mercure Mont Saint Michel,8.4,€ 139
1,Mont Saint-Michel,La Confiance,7.5,€ 200
2,Mont Saint-Michel,Le Relais Saint Michel,8.0,€ 250
3,Mont Saint-Michel,Hotel Gabriel,8.2,€ 75
4,Mont Saint-Michel,Le Mouton Blanc,7.2,€ 230


## 5. ETL : extraire, transformer, charger

### 5.1 Extraction
Les données sont relues **depuis le Data Lake** (S3).

In [12]:
cities_in = lake_read("raw", "cities")
weather_in = lake_read("raw", "weather")
hotels_in = lake_read("raw", "hotels")

print(len(cities_in), "villes |", len(weather_in), "lignes météo |", len(hotels_in), "hôtels")
print("\nValeurs manquantes dans les hôtels :")
print(hotels_in.isna().sum()[lambda s: s > 0])

35 villes | 35 lignes météo | 700 hôtels

Valeurs manquantes dans les hôtels :
hotel_score    5
dtype: int64


### 5.2 Transformation : nettoyer et organiser en 3 tables

On passe de 3 fichiers bruts à **3 tables liées** par `city_id`, adaptées à une base relationnelle :

```
cities (1) ──< weather_forecast (7 lignes par ville : un score par jour)
cities (1) ──< hotels           (jusqu'à 20 lignes par ville)
```

| Table | Règles de nettoyage |
|---|---|
| `cities` | on garde les villes géocodées avec succès, sans doublon ; colonnes renommées |
| `weather_forecast` | passage du format large (`weather_day_1…7`) au format **long** : une ligne par ville et par jour |
| `hotels` | prix `'€ 249'` converti en nombre ; URL nettoyée des paramètres de suivi ; doublons ville + URL supprimés ; note hors 0-10 mise à vide |

In [13]:
# ---------- Table cities ----------
cities = (cities_in[cities_in["status"] == "Ok"]
          .rename(columns={"place_id": "city_id", "place_name": "city_name",
                           "query": "search_query", "place_lat": "latitude",
                           "place_lon": "longitude"})
          .dropna(subset=["city_id", "latitude", "longitude"])
          .drop_duplicates("city_id")
          [["city_id", "city_name", "search_query", "latitude", "longitude"]]
          .reset_index(drop=True))

# ---------- Table weather_forecast (format long) ----------
day_cols = [c for c in weather_in.columns if c.startswith("weather_day_")]

weather_forecast = (weather_in[["place_id", *day_cols]].rename(columns={"place_id": "city_id"})
                    .melt(id_vars="city_id", value_vars=day_cols,
                          var_name="forecast_day", value_name="weather_score")
                    .dropna(subset=["weather_score"]))

weather_forecast["forecast_day"] = weather_forecast["forecast_day"].str.replace("weather_day_", "").astype(int)

weather_forecast = (weather_forecast[weather_forecast["city_id"].isin(cities["city_id"])]
                    .sort_values(["city_id", "forecast_day"]).reset_index(drop=True))


# ---------- Table hotels ----------
def parse_price(valeur):
    # '€\xa01 249' -> 1249 (les centimes éventuels sont ignorés)
    chiffres = re.sub(r"\D", "", re.sub(r"[.,]\d{1,2}$", "", re.sub(r"[^\d.,]", "", str(valeur))))
    return int(chiffres) if chiffres else None


hotels = hotels_in.copy()
hotels["price_eur"] = hotels["hotel_price"].map(parse_price).astype("Int64")
hotels["hotel_url"] = hotels["hotel_url"].str.split("?").str[0]       # retire les paramètres de suivi
hotels = (hotels.dropna(subset=["city_id", "hotel_name", "hotel_url"])
          .loc[lambda d: d["city_id"].isin(cities["city_id"])]         # ville connue seulement
          .drop_duplicates(["city_id", "hotel_url"]))                  # pas de doublon
hotels.loc[~hotels["hotel_score"].between(0, 10), "hotel_score"] = None  # note invalide -> vide
hotels = hotels.rename(columns={"hotel_lat": "latitude", "hotel_lon": "longitude"})
hotels = hotels[["city_id", "hotel_name", "hotel_url", "latitude", "longitude",
                 "hotel_score", "price_eur", "hotel_description"]].reset_index(drop=True)
hotels.insert(0, "hotel_id", hotels.index + 1)                         # clé primaire

print(f"cities : {len(cities)} | weather_forecast : {len(weather_forecast)} | hotels : {len(hotels)}")
hotels.head(3)

cities : 35 | weather_forecast : 245 | hotels : 700


,hotel_id,city_id,hotel_name,hotel_url,latitude,longitude,hotel_score,price_eur,hotel_description
0,1,283872868,Mercure Mont Saint Michel,https://www.booking.com/hotel/fr/mont-saint-mi...,48.635954,-1.51146,8.4,139,Chambre Lits Jumeaux Standard 2 lits simples A...
1,2,283872868,La Confiance,https://www.booking.com/hotel/fr/les-terrasses...,48.635954,-1.51146,7.5,200,Chambre Double ou Lits Jumeaux - Bâtiment Prin...
2,3,283872868,Le Relais Saint Michel,https://www.booking.com/hotel/fr/le-relais-sai...,48.635954,-1.51146,8.0,250,Classic Double or Twin Room Garden View Types ...


**Contrôles de qualité** : on vérifie ce que l'on attend avant de charger, plutôt que de découvrir un problème dans la base.

In [14]:
# Chaque contrôle doit réussir, sinon le notebook s'arrête ici (assert)
assert cities["city_id"].is_unique, "doublons dans cities"
assert (weather_forecast.groupby("city_id").size() == 7).all(), "une ville n'a pas 7 jours de météo"
assert weather_forecast["weather_score"].between(0, 100).all(), "score météo hors 0-100"
assert hotels["city_id"].isin(cities["city_id"]).all(), "hôtel sans ville"
assert hotels["hotel_score"].dropna().between(0, 10).all(), "note hors 0-10"
assert (hotels["price_eur"].dropna() > 0).all(), "prix négatif ou nul"

print("Tous les contrôles sont passés.")
print(f"{hotels['hotel_score'].isna().sum()} hôtels sans note, "
      f"{hotels['price_eur'].isna().sum()} sans prix (conservés, valeurs vides).")

Tous les contrôles sont passés.
5 hôtels sans note, 0 sans prix (conservés, valeurs vides).


### 5.3 Fichier enrichi pour les équipes métiers

Le livrable de l'énoncé est **un seul CSV croisant météo et hôtels** : une ligne par hôtel, avec sa ville, le score météo de la ville et le **rang** de la ville (1 = meilleure météo). Il est déposé dans la couche `enriched/` du Data Lake (local et S3).

In [15]:
# Score météo moyen et rang de chaque ville (1 = meilleure météo)
ranking = weather_forecast.groupby("city_id", as_index=False)["weather_score"].mean()
ranking["weather_rank"] = ranking["weather_score"].rank(method="min", ascending=False).astype(int)

enriched = (hotels
            .rename(columns={"latitude": "hotel_lat", "longitude": "hotel_lon"})
            .merge(cities[["city_id", "city_name", "latitude", "longitude"]]
                   .rename(columns={"latitude": "city_lat", "longitude": "city_lon"}), on="city_id")
            .merge(ranking, on="city_id")
            .sort_values(["weather_rank", "hotel_score"], ascending=[True, False])
            .reset_index(drop=True))

lake_write(enriched, "enriched", "kayak_enriched", "kayak_enriched.csv")
enriched[["weather_rank", "city_name", "weather_score", "hotel_name", "hotel_score", "price_eur"]].head(5)

  déposé en local et sur S3 : enriched/kayak_enriched/date=2026-11-02/kayak_enriched.csv


,weather_rank,city_name,weather_score,hotel_name,hotel_score,price_eur
0,1,Biarritz,73.571429,Chambre d'hôtes à Biarritz,9.4,69
1,1,Biarritz,73.571429,Maison Resaurie - Hôtel,9.4,210
2,1,Biarritz,73.571429,Hôtel de La Plage,9.3,206
3,1,Biarritz,73.571429,Hôtel & Espace Bien-être La Maison du Lierre,9.2,105
4,1,Biarritz,73.571429,Le Talaia Hôtel & Spa Biarritz - MGallery Coll...,9.0,198


## 6. Le Data Warehouse (SQL) : Neon et copie locale

Le **Data Warehouse** contient les données **nettoyées et structurées**, prêtes à être interrogées en SQL par les équipes d'analyse.

- **Neon** : base **PostgreSQL serverless**, c'est la base de référence. La connexion est chiffrée en **SSL** (`sslmode=require` dans la chaîne de connexion) et l'identifiant est lu dans `.env`. Neon met le calcul en veille quand il n'est pas utilisé : la première connexion peut prendre quelques secondes, d'où `pool_pre_ping=True` qui évite d'utiliser une connexion périmée.
- **SQLite local** (`kayak_warehouse.db`) : **copie de sauvegarde**, chargée **dans la même étape** avec exactement le même schéma et les mêmes données.

Le schéma impose des **clés primaires, des clés étrangères et des contraintes** (note entre 0 et 10, score météo entre 0 et 100...) : la base refuse elle-même une donnée incohérente. Une **vue** `city_weather_summary` calcule le score moyen et le rang de chaque ville.

In [16]:
# --- Connexions : Neon (référence) + SQLite (copie locale) ---
# On force le pilote psycopg2 (selon la version de SQLAlchemy, le pilote par défaut diffère)
neon_url = re.sub(r"^postgres(ql)?://", "postgresql+psycopg2://", os.environ["NEON_DATABASE_URL"])
neon_engine = create_engine(neon_url, pool_pre_ping=True)      # SSL imposé par la chaîne Neon

data_dir = Path("../data_warehouse")
data_dir.mkdir(parents=True, exist_ok=True)

sqlite_path = data_dir / "kayak_warehouse.db"
local_engine = create_engine(f"sqlite:///{sqlite_path.as_posix()}")

ENGINES = {"Neon": neon_engine, "SQLite local": local_engine}

# --- Schéma : on repart de zéro pour que le notebook soit rejouable ---
SCHEMA = [
    "DROP VIEW IF EXISTS city_weather_summary",
    "DROP TABLE IF EXISTS enriched_data",
    "DROP TABLE IF EXISTS hotels",
    "DROP TABLE IF EXISTS weather_forecast",
    "DROP TABLE IF EXISTS cities",
    '''CREATE TABLE cities (
        city_id      BIGINT PRIMARY KEY,
        city_name    TEXT NOT NULL,
        search_query TEXT NOT NULL,
        latitude     DOUBLE PRECISION NOT NULL CHECK (latitude BETWEEN -90 AND 90),
        longitude    DOUBLE PRECISION NOT NULL CHECK (longitude BETWEEN -180 AND 180))''',
    '''CREATE TABLE weather_forecast (
        city_id       BIGINT NOT NULL REFERENCES cities (city_id),
        forecast_day  SMALLINT NOT NULL CHECK (forecast_day BETWEEN 1 AND 7),
        weather_score NUMERIC(5, 2) NOT NULL CHECK (weather_score BETWEEN 0 AND 100),
        PRIMARY KEY (city_id, forecast_day))''',
    '''CREATE TABLE hotels (
        hotel_id          INTEGER PRIMARY KEY,
        city_id           BIGINT NOT NULL REFERENCES cities (city_id),
        hotel_name        TEXT NOT NULL,
        hotel_url         TEXT NOT NULL,
        latitude          DOUBLE PRECISION,
        longitude         DOUBLE PRECISION,
        hotel_score       NUMERIC(3, 1) CHECK (hotel_score BETWEEN 0 AND 10),
        price_eur         INTEGER CHECK (price_eur > 0),
        hotel_description TEXT,
        UNIQUE (city_id, hotel_url))''',
    '''CREATE TABLE enriched_data (
        hotel_id          INTEGER PRIMARY KEY REFERENCES hotels (hotel_id),
        city_id           BIGINT NOT NULL REFERENCES cities (city_id),
        hotel_name        TEXT NOT NULL,
        hotel_url         TEXT NOT NULL,
        hotel_lat         DOUBLE PRECISION,
        hotel_lon         DOUBLE PRECISION,
        hotel_score       NUMERIC(3, 1) CHECK (hotel_score BETWEEN 0 AND 10),
        price_eur         INTEGER CHECK (price_eur > 0),
        hotel_description TEXT,
        city_name         TEXT NOT NULL,
        city_lat          DOUBLE PRECISION NOT NULL,
        city_lon          DOUBLE PRECISION NOT NULL,
        weather_score     NUMERIC(5, 2) NOT NULL CHECK (weather_score BETWEEN 0 AND 100),
        weather_rank      INTEGER NOT NULL CHECK (weather_rank > 0))''',
    '''CREATE VIEW city_weather_summary AS
       SELECT c.city_id, c.city_name, c.latitude, c.longitude,
              ROUND(AVG(w.weather_score), 2) AS weather_score,
              RANK() OVER (ORDER BY ROUND(AVG(w.weather_score), 2) DESC) AS weather_rank
       FROM cities c JOIN weather_forecast w USING (city_id)
       GROUP BY c.city_id, c.city_name, c.latitude, c.longitude''',
]

# --- Chargement : chaque base dans une seule transaction (tout ou rien) ---
for name, engine in ENGINES.items():
    with engine.begin() as conn:
        for statement in SCHEMA:
            conn.execute(text(statement))
        for table_name, table in [("cities", cities), ("weather_forecast", weather_forecast), ("hotels", hotels), ("enriched_data", enriched)]:
            table.to_sql(table_name, conn, if_exists="append", index=False)   # cities d'abord (clés étrangères)
    print(f"Chargement terminé : {name}")


def sql(query, engine=neon_engine):
    # Exécute une requête SQL (par défaut sur Neon) et renvoie un DataFrame.
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)


# --- Validation : les volumes de CHAQUE base doivent égaler ceux des DataFrames ---
expected = {"cities": len(cities), "weather_forecast": len(weather_forecast),
            "hotels": len(hotels), "enriched_data": len(enriched)}
for name, engine in ENGINES.items():
    counts = sql('''SELECT 'cities' AS table_name, COUNT(*) AS n FROM cities
                    UNION ALL SELECT 'weather_forecast', COUNT(*) FROM weather_forecast
                    UNION ALL SELECT 'hotels', COUNT(*) FROM hotels
                    UNION ALL SELECT 'enriched_data', COUNT(*) FROM enriched_data''', engine)
    assert counts.set_index("table_name")["n"].to_dict() == expected, f"volumes incorrects dans {name}"
print("Volumes identiques dans Neon et en local :", expected)

Chargement terminé : Neon
Chargement terminé : SQLite local
Volumes identiques dans Neon et en local : {'cities': 35, 'weather_forecast': 245, 'hotels': 700, 'enriched_data': 700}


**Les analystes peuvent maintenant interroger Neon en SQL.**.

In [17]:
# En cas d'ex æquo, on départage par city_id : le classement est toujours le même
top5_sql = sql('''SELECT city_id, city_name, weather_score, weather_rank
                  FROM city_weather_summary ORDER BY weather_rank, city_id LIMIT 5''')
top5_sql

,city_id,city_name,weather_score,weather_rank
0,290313639,Biarritz,73.57,1
1,290541150,Bayonne,73.57,1
2,102315906,Lille,70.71,3
3,99155202,Amiens,69.29,4
4,81776233,Bormes-les-Mimosas,67.14,5


In [18]:
# Les 5 villes sont celles de la requête précédente ; en cas d'ex æquo (note, prix), on départage par hotel_id
top_hotels_sql = sql('''SELECT s.city_name, h.hotel_id, h.hotel_name, h.hotel_score, h.price_eur
                        FROM hotels h JOIN city_weather_summary s USING (city_id)
                        WHERE h.city_id IN (SELECT city_id FROM city_weather_summary
                                            ORDER BY weather_rank, city_id LIMIT 5)
                          AND h.hotel_score IS NOT NULL
                        ORDER BY h.hotel_score DESC, (h.price_eur IS NULL), h.price_eur, h.hotel_id
                        LIMIT 20''')
top_hotels_sql

,city_name,hotel_id,hotel_name,hotel_score,price_eur
0,Bormes-les-Mimosas,368,Bohème 375,9.8,216
1,Amiens,138,"""Au jardin sur l'eau"" Hortillonnages",9.5,131
2,Bormes-les-Mimosas,380,Studio Bormes,9.5,260
3,Biarritz,655,Chambre d'hôtes à Biarritz,9.4,69
4,Amiens,140,Maison Longuevie,9.4,152
5,Bormes-les-Mimosas,376,Appartement entièrement climatisé avec spa,9.4,167
6,Biarritz,660,Maison Resaurie - Hôtel,9.4,210
7,Biarritz,651,Hôtel de La Plage,9.3,206
8,Bormes-les-Mimosas,365,Les Terrasses de Lô,9.2,102
9,Biarritz,647,Hôtel & Espace Bien-être La Maison du Lierre,9.2,105


## 7. Restitution : deux cartes

1. **Top 5 des destinations** : les 5 villes avec le meilleur score météo moyen sur 7 jours.
2. **Top 20 des hôtels** dans ces 5 villes : classés par note décroissante, puis par prix croissant en cas d'égalité (même règle que la requête SQL ci-dessus).

Les cartes lisent le **fichier enrichi** (identique à la base). Elles sont aussi enregistrées en HTML dans `outputs/maps/`.

### 7.1 Carte 1 : Top 5 des destinations

In [19]:
MAPS_DIR = Path("../outputs/maps")
MAPS_DIR.mkdir(parents=True, exist_ok=True)

CENTRE_FRANCE = {"lat": 46.6, "lon": 2.4}

top_cities = (enriched.drop_duplicates("city_id")
              .sort_values(["weather_rank", "city_id"]).head(5))      # même départage que le SQL

# Contrôle : mêmes 5 villes dans le fichier enrichi et dans la base
assert top_cities["city_name"].tolist() == top5_sql["city_name"].tolist(), "écart SQL / fichier (villes)"
fig1 = px.scatter_map(top_cities, lat="city_lat", lon="city_lon", hover_name="city_name",
                      hover_data={"weather_score": ":.1f", "city_lat": False, "city_lon": False},
                      color="weather_score", size="weather_score",
                      color_continuous_scale="RdYlGn", center=CENTRE_FRANCE, zoom=4.5, height=600)
fig1.update_layout(map_style="open-street-map", title="Top 5 des destinations (météo à 7 jours)",
                   coloraxis_colorbar_title="Score météo")
fig1.write_html(MAPS_DIR / "top5_destinations.html")
fig1.show()

### 7.2 Le Top 20 des hôtels et leurs vraies coordonnées

La page de résultats de Booking ne donne pas la position de chaque hôtel : tous les hôtels d'une ville auraient les coordonnées de la ville et seraient **superposés** sur la carte. Pour l'éviter sans multiplier les requêtes, on visite **uniquement la page des 20 hôtels retenus** et on lit leurs vraies coordonnées (même politesse que pour le scraping : pauses, arrêt après 3 échecs). Si une page échoue, l'hôtel garde les coordonnées de sa ville.

> Ces coordonnées servent à la **carte uniquement** : la base et le fichier enrichi conservent celles de la ville.

In [20]:
# --- Top 20 : mêmes règles que la requête SQL ---
top_hotels = (enriched[enriched["city_id"].isin(top_cities["city_id"])]
              .dropna(subset=["hotel_score"])
              .sort_values(["hotel_score", "price_eur", "hotel_id"], ascending=[False, True, True])
              .head(20)
              .reset_index(drop=True)
              .copy())

# Contrôle : la carte (fichier enrichi) et la requête SQL donnent les mêmes hôtels
if top_hotels["hotel_id"].tolist() != top_hotels_sql["hotel_id"].tolist():
    print(pd.concat([top_hotels[["hotel_id", "hotel_name", "hotel_score", "price_eur"]].add_prefix("fichier_"),
                     top_hotels_sql[["hotel_id", "hotel_name", "hotel_score", "price_eur"]].add_prefix("sql_")], axis=1))
    raise AssertionError("écart SQL / fichier : voir le tableau ci-dessus")


def parse_latlng(html):
    # Lit la position d'un hôtel dans sa page : attribut data-atlas-latlng ("48.85,2.35").
    el = BeautifulSoup(html, "html.parser").select_one("[data-testid='map-entry-point-desktop-wrapper']")
    if el is None or not el.get("data-atlas-latlng"):
        return None
    lat, lon = (float(x) for x in el["data-atlas-latlng"].split(","))
    return lat, lon


def fetch_real_coordinates(df):
    # Remplace les coordonnées (ville) par les vraies coordonnées de chaque hôtel du DataFrame.
    options = webdriver.ChromeOptions()
    options.add_argument("--lang=fr-FR")
    driver = webdriver.Chrome(options=options)
    df, failures = df.copy(), 0
    try:
        for i, row in df.iterrows():
            try:
                driver.get(row["hotel_url"])
                WebDriverWait(driver, 10).until(EC.presence_of_element_located(
                    (By.CSS_SELECTOR, "[data-testid='map-entry-point-desktop-wrapper']")))
                coords = parse_latlng(driver.page_source)
                if coords is None:
                    raise ValueError("coordonnées absentes de la page")
                df.loc[i, ["hotel_lat", "hotel_lon"]] = list(coords)
                failures = 0
            except Exception as err:
                failures += 1
                print(f"Coordonnées non trouvées pour {row['hotel_name']} : {type(err).__name__}")
                if failures >= 3:
                    print("Blocage probable : les hôtels restants gardent les coordonnées de leur ville.")
                    break
            time.sleep(random.uniform(3, 7))
    finally:
        driver.quit()
    return df

In [21]:
top_hotels = fetch_real_coordinates(top_hotels)      # environ 2 minutes pour 20 hôtels
n_precis = (top_hotels["hotel_lat"] != top_hotels["city_lat"]).sum()
print(f"{n_precis} hôtels sur {len(top_hotels)} avec leurs vraies coordonnées")

20 hôtels sur 20 avec leurs vraies coordonnées


### 7.3 Carte 2 : Top 20 des hôtels

In [22]:
fig2 = px.scatter_map(top_hotels, lat="hotel_lat", lon="hotel_lon", hover_name="hotel_name",
                      hover_data={"city_name": True, "hotel_score": ":.1f", "price_eur": True,
                                  "hotel_lat": False, "hotel_lon": False},
                      color="hotel_score", color_continuous_scale="Viridis",
                      center=CENTRE_FRANCE, zoom=4.5, height=600)
fig2.update_traces(marker={"size": 14})
fig2.update_layout(map_style="open-street-map", title="Top 20 des hôtels (note Booking.com)",
                   coloraxis_colorbar_title="Note")
fig2.write_html(MAPS_DIR / "top20_hotels.html")
fig2.show()

top_hotels[["city_name", "hotel_name", "hotel_score", "price_eur"]]

,city_name,hotel_name,hotel_score,price_eur
0,Bormes-les-Mimosas,Bohème 375,9.8,216
1,Amiens,"""Au jardin sur l'eau"" Hortillonnages",9.5,131
2,Bormes-les-Mimosas,Studio Bormes,9.5,260
3,Biarritz,Chambre d'hôtes à Biarritz,9.4,69
4,Amiens,Maison Longuevie,9.4,152
5,Bormes-les-Mimosas,Appartement entièrement climatisé avec spa,9.4,167
6,Biarritz,Maison Resaurie - Hôtel,9.4,210
7,Biarritz,Hôtel de La Plage,9.3,206
8,Bormes-les-Mimosas,Les Terrasses de Lô,9.2,102
9,Biarritz,Hôtel & Espace Bien-être La Maison du Lierre,9.2,105


## 8. Bilan

**Résultats** : Data Lake en 2 couches, sauvegardé en local et sur S3 ; Data Warehouse de 4 tables contrôlées, chargé dans Neon et en copie locale SQLite ; 2 cartes (Top 5 destinations, Top 20 hôtels).

**Limites connues**
- **Score météo** : il repose sur le seul code météo. La température, les précipitations et le vent pourraient l'affiner.
- **Coordonnées des hôtels** : celles de la ville dans la base et le fichier enrichi ; les vraies coordonnées ne sont lues que pour le Top 20, pour la carte.
- **Descriptions** : le texte de la page de résultats décrit les chambres proposées, pas l'hôtel.
- **Dates** : prix, notes et prévisions correspondent au moment de la collecte (séjour simulé de 1 nuit à partir de J+1). Chaque exécution remplace les tables de Neon.
- **Scraping** : usage pédagogique uniquement, voir §3. En production : partenariat ou API officielle de Booking.com, abonnement Open-Meteo.

**Pistes** : enrichir le score météo (température, pluie), chargement incrémental plutôt que complet, durée de conservation de la couche `raw/`, exécution planifiée.

*Sources : coordonnées © contributeurs OpenStreetMap (ODbL) via Nominatim · météo : Open-Meteo.com · hôtels : Booking.com.*